# Ground-Truth Data Generation

Generates ~5 user-style questions per zoning record with an LLM, producing
`data/ground-truth-retrieval.csv` (columns: `id`, `question`).

An offline template-based fallback (no API key needed) is available at
`data/generate_ground_truth_offline.py`; the repo ships with its output so
the retrieval evaluation runs out of the box. Re-running this notebook
replaces it with LLM-generated questions.

In [ ]:
import json

import pandas as pd
from openai import OpenAI
from tqdm.auto import tqdm

client = OpenAI()

df = pd.read_csv("../data/zoning.csv")
documents = df.to_dict(orient="records")
len(documents)

In [ ]:
prompt_template = """
You emulate a user of our property & zoning research assistant.
Formulate 5 questions this user might ask based on the provided zoning
code section. Make the questions specific to this section's content.
The record should contain the answer to the questions, and the questions
should be complete, not too short, and use the words a homeowner or
developer would actually use (including common synonyms like "granny
flat" for accessory dwelling unit) rather than copying the legal text
verbatim.

The record:

section: {section}
district: {district}
category: {category}
title: {title}
text: {text}

Provide the output in parsable JSON without using code blocks:

{{"questions": ["question1", "question2", ..., "question5"]}}
""".strip()

In [ ]:
def generate_questions(doc):
    prompt = prompt_template.format(**doc)
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
    )
    return response.choices[0].message.content

In [ ]:
results = {}

for doc in tqdm(documents):
    doc_id = doc["id"]
    if doc_id in results:
        continue
    raw = generate_questions(doc)
    results[doc_id] = json.loads(raw)["questions"]

In [ ]:
final_results = [
    (doc_id, q)
    for doc_id, questions in results.items()
    for q in questions
]

df_results = pd.DataFrame(final_results, columns=["id", "question"])
df_results.to_csv("../data/ground-truth-retrieval.csv", index=False)
df_results.head()